# Paso 1 · Carga y verificación del dataset (Lending Club)

**Objetivo de este notebook:** comprobar que Colab ve tu archivo, que se carga *completo* (sin muestreo, como exige el enunciado) y entender qué trae la columna `loan_status`, que es la base de la variable objetivo `default`.

**Por qué empezamos aquí:** antes de modelar hay que saber exactamente con cuántas filas trabajamos y qué categorías tiene el target. De eso dependen todas las decisiones posteriores.

Las celdas se ejecutan en orden.

## 1. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Ubicar el archivo
Ajusta `RUTA` si tu carpeta de Drive se llama distinto. Si lo que subiste es un `.zip`, descomenta la línea de `unzip`.

In [ ]:
import os, glob

RUTA = '/content/drive/MyDrive/ML_LendingClub'   # <- cambia esto si tu carpeta se llama distinto

print('Contenido de la carpeta:')
for f in sorted(os.listdir(RUTA)):
    tam = os.path.getsize(os.path.join(RUTA, f)) / 1e6
    print(f'  {f}  ({tam:,.0f} MB)')

# Si subiste el .zip de Kaggle, descomenta la siguiente línea (solo la primera vez):
!unzip -n "{RUTA}"/*.zip -d "{RUTA}"

Contenido de la carpeta:
  archive.zip  (1,357 MB)
Archive:  /content/drive/MyDrive/ML_LendingClub/archive.zip
  inflating: /content/drive/MyDrive/ML_LendingClub/accepted_2007_to_2018Q4.csv.gz  
  inflating: /content/drive/MyDrive/ML_LendingClub/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv  
  inflating: /content/drive/MyDrive/ML_LendingClub/rejected_2007_to_2018Q4.csv.gz  
  inflating: /content/drive/MyDrive/ML_LendingClub/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv  


In [ ]:
archivos = sorted(glob.glob(RUTA + '/*accepted*'))
print('Archivos candidatos (aceptados):', archivos)

Archivos candidatos (aceptados): []


## 3. Declarar el hardware
El enunciado pide declarar núcleos y memoria para que la comparación de tiempos sea interpretable. Lo dejamos registrado desde ya.

In [ ]:
import os, psutil
print('Núcleos (CPU):', os.cpu_count())
print('RAM total (GB):', round(psutil.virtual_memory().total / 1e9, 1))

Núcleos (CPU): 8
RAM total (GB): 54.8


## 4. Cargar el CSV completo
`low_memory=False` hace que pandas lea las columnas con tipos consistentes (evita advertencias de tipos mixtos). Medimos el tiempo de carga porque el enunciado permite compararlo con Spark.

In [ ]:
import pandas as pd, time

t0 = time.time()
df = pd.read_csv(archivos[0], low_memory=False)
t_carga = time.time() - t0

print(f'Tiempo de carga con pandas: {t_carga:,.1f} s')
print('Dimensión (filas, columnas):', df.shape)

Tiempo de carga con pandas: 50.7 s
Dimensión (filas, columnas): (2260701, 151)


In [ ]:
import glob, os
for f in sorted(glob.glob(RUTA + '/**/*', recursive=True)):
    print(f.replace(RUTA, ''), f'({os.path.getsize(f)/1e6:,.0f} MB)' if os.path.isfile(f) else '(carpeta)')

/archive.zip (1,357 MB)


In [ ]:
archivos = sorted(f for f in glob.glob(RUTA + '/*accepted*') if os.path.isfile(f))
print(archivos)

['/content/drive/MyDrive/ML_LendingClub/accepted_2007_to_2018Q4.csv.gz']


## 5. Revisar `loan_status`
Esta columna es el origen del target. El enunciado define `default = 1` si `loan_status == "Charged Off"` y `0` en otro caso. Hay que ver **qué otras categorías existen** (por ejemplo, préstamos aún vigentes) porque cambian lo que significa el 0.

In [ ]:
conteo = df['loan_status'].value_counts(dropna=False)
print(conteo)
print()
print('Total filas:', len(df))
print('Fully Paid + Charged Off:', int(conteo.get('Fully Paid', 0) + conteo.get('Charged Off', 0)))

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

Total filas: 2260701
Fully Paid + Charged Off: 1345310


In [ ]:
print('Columna id presente:', 'id' in df.columns)
print('Primeras 5 columnas:', list(df.columns[:5]))
df[['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'purpose', 'home_ownership', 'addr_state', 'loan_status']].head(5)

Columna id presente: True
Primeras 5 columnas: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv']


,loan_amnt,int_rate,annual_inc,dti,purpose,home_ownership,addr_state,loan_status
0,3600.0,13.99,55000.0,5.91,debt_consolidation,MORTGAGE,PA,Fully Paid
1,24700.0,11.99,65000.0,16.06,small_business,MORTGAGE,SD,Fully Paid
2,20000.0,10.78,63000.0,10.78,home_improvement,MORTGAGE,IL,Fully Paid
3,35000.0,14.85,110000.0,17.06,debt_consolidation,MORTGAGE,NJ,Current
4,10400.0,22.45,104433.0,25.37,major_purchase,MORTGAGE,PA,Fully Paid
